# 02 — QLoRA Fine-tuning

Fine-tune Qwen3-1.7B on the ICD-10 training data using QLoRA.

## ⚠️ Requires GPU

This notebook needs an **L4** (preferred) or T4 High-RAM GPU. Switch runtime:
`Runtime → Change runtime type → L4 GPU → High-RAM`.

## What happens

1. Load train/val/test from `data/processed/`
2. Load Qwen3-1.7B in 4-bit with QLoRA adapter
3. SFT training with TRL's `SFTTrainer`
4. Save LoRA adapter → merge into FP16 weights → save merged model

## Outputs
- `models/adapter/` — just the LoRA delta weights (~35 MB)
- `models/merged-fp16/` — full fine-tuned model in fp16 (~3.4 GB)

## Runtime
~45-60 min on L4, ~90 min on T4 High-RAM. ~5-7 compute units.

## Training config (sized for ~40k examples, 1,564 codes)

| Setting | Value |
|---|---|
| Batch size (per device) | 8 |
| Gradient accumulation | 4 (effective batch = 32) |
| Epochs | 3 |
| Learning rate | 2e-4 (cosine schedule) |
| LoRA rank | 16 |
| Max sequence length | 512 |
| Eval every | 250 steps |



## 0. Bootstrap + GPU check

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = (
    'COLAB_GPU' in os.environ
    or 'COLAB_RELEASE_TAG' in os.environ
    or (Path('/content').exists() and not Path('/content').is_symlink())
)

if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/icd10-slm')
else:
    PROJECT = Path.cwd()
    while not (PROJECT / 'requirements.txt').exists():
        if PROJECT.parent == PROJECT:
            raise RuntimeError(f"Couldn't find repo root from {Path.cwd()}")
        PROJECT = PROJECT.parent

assert PROJECT.exists(), f"Project not found at {PROJECT}"
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Load secrets — .env from /content (Path B) or from project
if not os.getenv('HF_TOKEN'):
    try:
        from dotenv import load_dotenv
    except ImportError:
        !pip install -q python-dotenv
        from dotenv import load_dotenv
    import shutil
    candidates = [
        Path('/content/.env'),
        Path('/content/drive/MyDrive/icd10-slm/.env'),
    ]
    for c in candidates:
        if c.exists():
            if c != Path('/content/.env'):
                shutil.copy(c, '/content/.env')
            load_dotenv('/content/.env')
            break

assert os.getenv('HF_TOKEN'), "HF_TOKEN missing"
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

from src import config
print(f"✓ env:     {'Colab' if IN_COLAB else 'Local'}")
print(f"✓ project: {PROJECT}")
print(f"✓ model:   {config.BASE_MODEL}")


In [ ]:
import torch
assert torch.cuda.is_available(), "GPU required. Change runtime type to L4 or T4 High-RAM."

device = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"✓ GPU: {device}  |  VRAM: {vram_gb:.1f} GB")

# Warn about limited-VRAM scenarios
if vram_gb < 14:
    print("⚠ Limited VRAM — will reduce batch size")


## 1. Load the data

Reload the JSONL files produced by Notebook 1. We use HuggingFace `datasets`
for compatibility with `SFTTrainer`.


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    'json',
    data_files={
        'train': str(config.TRAIN_PATH),
        'val':   str(config.VAL_PATH),
        'test':  str(config.TEST_PATH),
    },
)
print(ds)

# Peek at one example
print("\n--- sample training example ---")
print(ds['train'][0]['text'][:500])


## 2. Load model + tokenizer in 4-bit (QLoRA)

We load Qwen3-1.7B with BitsAndBytes 4-bit quantization, then attach a LoRA
adapter. This is the *Q* in Q**LoRA — base weights stay frozen in 4-bit,
only the small LoRA matrices train.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False  # required for training

print(f"✓ loaded in 4-bit, ~{sum(p.numel() for p in model.parameters())/1e9:.2f}B params")
print(f"  VRAM used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## 3. Test base model before training

Let's see what Qwen3-1.7B says before fine-tuning, for 3 examples from
the test set. Baseline we'll compare against after training.


In [ ]:
from transformers import pipeline

def predict(msg, model_obj, tok, max_new=16):
    messages = [
        {"role": "system", "content": config.SYSTEM_PROMPT},
        {"role": "user",   "content": msg},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(model_obj.device)
    with torch.no_grad():
        out = model_obj.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, pad_token_id=tok.pad_token_id,
        )
    # Trim off the prompt; return only the generation
    gen = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return gen.strip()

print("BASE MODEL PREDICTIONS (before training)")
print("=" * 60)
for ex in ds['test'].select(range(3)):
    pred = predict(ex['clinical_text'], model, tokenizer)
    hit = pred.strip().upper() == ex['code'].strip().upper()
    print(f"Input:  {ex['clinical_text'][:60]}")
    print(f"Gold:   {ex['code']}")
    print(f"Pred:   {pred!r}  {'✓' if hit else '✗'}")
    print("-" * 60)


## 4. Configure LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=config.LORA_R,
    lora_alpha=config.LORA_ALPHA,
    lora_dropout=config.LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    # Qwen3 attention + MLP projection layers
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


## 5. Training arguments + trainer

In [ ]:
from trl import SFTTrainer, SFTConfig

# Scale down batch size if low VRAM
batch_size = 8 if vram_gb >= 14 else 4

args = SFTConfig(
    output_dir=str(config.ADAPTER_DIR.parent / 'training_run'),

    # Batch
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=4,

    # Schedule
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,

    # Sequence formatting (SFTConfig expects these here, not in Trainer)
    max_seq_length=config.MAX_SEQ_LENGTH,
    dataset_text_field='text',
    packing=False,

    # Precision
    bf16=True,
    optim='paged_adamw_8bit',

    # Logging / eval
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=250,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Reproducibility
    seed=42,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds['train'],
    eval_dataset=ds['val'],
    processing_class=tokenizer,
)

print(f"✓ trainer ready")
print(f"  batch size (per dev):     {args.per_device_train_batch_size}")
print(f"  grad accum:               {args.gradient_accumulation_steps}")
print(f"  effective batch:          {args.per_device_train_batch_size * args.gradient_accumulation_steps}")
print(f"  total training steps:     ~{len(ds['train']) * args.num_train_epochs // (args.per_device_train_batch_size * args.gradient_accumulation_steps):,}")


## 6. Train 🚀

Wall-clock: ~45-60 min on L4, ~90 min on T4. Eval runs every 250 steps so you
can watch val loss drop. If it plateaus or rises → early stopping kicks in via
`load_best_model_at_end`.

**Don't click into the notebook or switch tabs during training** — Colab may
demote the runtime.


In [ ]:
import time
t0 = time.time()
trainer.train()
print(f"\n✓ training done in {(time.time()-t0)/60:.1f} min")


## 7. Save the LoRA adapter

In [ ]:
config.ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(config.ADAPTER_DIR))
tokenizer.save_pretrained(str(config.ADAPTER_DIR))

adapter_size_mb = sum(
    f.stat().st_size for f in config.ADAPTER_DIR.rglob('*') if f.is_file()
) / 1024 / 1024
print(f"✓ adapter saved to {config.ADAPTER_DIR}  ({adapter_size_mb:.1f} MB)")


## 8. Test fine-tuned model on the same 3 examples

Compare predictions to the base-model pre-training output (Section 3).


In [ ]:
print("FINE-TUNED MODEL PREDICTIONS")
print("=" * 60)
model.eval()

hits = 0
for ex in ds['test'].select(range(3)):
    pred = predict(ex['clinical_text'], model, tokenizer)
    hit = pred.strip().upper() == ex['code'].strip().upper()
    hits += hit
    print(f"Input:  {ex['clinical_text'][:60]}")
    print(f"Gold:   {ex['code']}")
    print(f"Pred:   {pred!r}  {'✓' if hit else '✗'}")
    print("-" * 60)
print(f"\n3-sample accuracy: {hits}/3  (full test eval in Notebook 3)")


## 9. Merge adapter into FP16 base → save for deployment

The adapter alone can't be deployed cleanly — we need to merge the LoRA delta
into the base weights. This produces a standalone FP16 model that Notebook 3
will quantize to Q4 GGUF.


In [ ]:
import torch, gc

# Free the 4-bit training model from VRAM
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print(f"  VRAM after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Reload base in FP16 (no quantization), then load adapter and merge
from transformers import AutoModelForCausalLM
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='cpu',  # merge on CPU to avoid VRAM pressure
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, str(config.ADAPTER_DIR))
merged = merged.merge_and_unload()

config.MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(config.MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(config.MERGED_DIR))

merged_size_mb = sum(
    f.stat().st_size for f in config.MERGED_DIR.rglob('*') if f.is_file()
) / 1024 / 1024
print(f"✓ merged FP16 model saved to {config.MERGED_DIR}")
print(f"  size: {merged_size_mb/1024:.2f} GB")


## ✅ Done

You now have:
- `models/adapter/` — LoRA delta (~35 MB)
- `models/merged-fp16/` — standalone FP16 model (~3.4 GB)

### Next

Switch runtime to **T4 or CPU** (quantization doesn't need L4) and open
`03_quantization_and_benchmarks.ipynb`.

### Commit adapter to git? No.

Both `models/adapter/` and `models/merged-fp16/` are in `.gitignore` —
they're too big, and regenerable. They live on Drive only.
